In [1]:
# --- Instalações ---
!pip install -q scikit-learn pandas numpy tensorflow imbalanced-learn

# --- Imports de Sistema ---
import pandas as pd
import numpy as np
import sys
import os
import warnings

# --- Imports de Deep Learning (Keras/TensorFlow) ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
# --- NOVOS IMPORTS DE CAMADA ---
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Embedding, Dropout, LSTM, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# --- Imports de Machine Learning (Sklearn) ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.exceptions import UndefinedMetricWarning

# --- Imports de Balanceamento ---
from imblearn.under_sampling import RandomUnderSampler

# Ignora warnings
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Montar o Google Drive ---
print("Montando Google Drive...")
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive já montado ou erro: {e}")

# --- Confirmação de GPU ---
print("\nVerificando GPU...")
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  print(
      '\n\nATENÇÃO: GPU NÃO ENCONTRADA! '
      'Vá em "Ambiente de execução" -> "Alterar o tipo de ambiente de execução" e selecione "GPU (T4)".'
  )
else:
  print(f'GPU encontrada: {device_name}')

Montando Google Drive...
Mounted at /content/drive

Verificando GPU...
GPU encontrada: /device:GPU:0


In [2]:
# --- 1. Configurações ---
ARQUIVO_ENTRADA = '/content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/dataset_FINAL_ACHATADO.csv'
TARGET_FUNCTION = 'protein binding'

# --- Configs da CNN/LSTM ---
MAX_SEQ_LENGTH = 2000 # Nosso "corte" de sequência
EMBEDDING_DIM = 100
VOCAB_SIZE = 16 + 1 # +1 para o padding/OOV (será atualizado)

In [3]:
print(f"Iniciando Pipeline Final (CNN-LSTM)...")
print(f"Arquivo de entrada: {ARQUIVO_ENTRADA}")
print(f"Tarefa: Classificação Binária (Target = '{TARGET_FUNCTION}')\n")

# --- 3.1 Carregar Dados ---
print("Carregando dataset 'achatado'...")
try:
    df_flat = pd.read_csv(ARQUIVO_ENTRADA)
except FileNotFoundError:
    print(f"ERRO: Arquivo '{ARQUIVO_ENTRADA}' não encontrado.")
    raise

if 'label' not in df_flat.columns or 'Sequencia' not in df_flat.columns:
    print(f"ERRO: O CSV de entrada não contém 'label' ou 'Sequencia'.")
    raise

print(f"Dataset carregado com {len(df_flat)} genes únicos.")

# --- 3.2 Balanceamento (Undersampling) ---
print(f"Distribuição ANTES do balanceamento:\n{df_flat['label'].value_counts()}\n")
print("Iniciando Undersampling para forçar balanço 1:1...")
rus = RandomUnderSampler(random_state=42)
X_para_amostrar = df_flat.index.values.reshape(-1, 1)
y_para_amostrar = df_flat['label']
X_res, y_res = rus.fit_resample(X_para_amostrar, y_para_amostrar)
indices_balanceados = X_res.flatten()
df_balanced = df_flat.loc[indices_balanceados].copy()
print(f"Dataset balanceado criado.\nDistribuição DEPOIS:\n{df_balanced['label'].value_counts()}\n")

# --- 3.3 Preparação para CNN (Tokenizer e Padding) ---
print("Preparando dados para CNN (Tokenizing e Padding)...")
df_balanced['Sequencia'] = df_balanced['Sequencia'].astype(str)
tokenizer = Tokenizer(char_level=True, lower=False, oov_token='U')
tokenizer.fit_on_texts(df_balanced['Sequencia'])
sequences_tokenized = tokenizer.texts_to_sequences(df_balanced['Sequencia'])

# Atualiza o tamanho do vocabulário
VOCAB_SIZE = len(tokenizer.word_index)
print(f"Tamanho do Vocabulário (A,T,C,G,N...): {VOCAB_SIZE}")

X_padded = pad_sequences(sequences_tokenized,
                         maxlen=MAX_SEQ_LENGTH,
                         padding='post',
                         truncating='post')
Y_labels = df_balanced['label'].values
print(f"Matriz de features X criada: {X_padded.shape}")
print(f"Vetor de labels Y criado: {Y_labels.shape}")

# --- 3.4 Divisão de Treino/Teste ---
print("\nDividindo dados (80% treino / 20% teste)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    Y_labels,
    test_size=0.2,
    random_state=42,
    stratify=Y_labels
)
print(f"Tamanho do Treino: {X_train.shape[0]}")
print(f"Tamanho do Teste: {X_test.shape[0]}")

Iniciando Pipeline Final (CNN-LSTM)...
Arquivo de entrada: /content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/dataset_FINAL_ACHATADO.csv
Tarefa: Classificação Binária (Target = 'protein binding')

Carregando dataset 'achatado'...
Dataset carregado com 5014 genes únicos.
Distribuição ANTES do balanceamento:
label
1    3723
0    1291
Name: count, dtype: int64

Iniciando Undersampling para forçar balanço 1:1...
Dataset balanceado criado.
Distribuição DEPOIS:
label
0    1291
1    1291
Name: count, dtype: int64

Preparando dados para CNN (Tokenizing e Padding)...
Tamanho do Vocabulário (A,T,C,G,N...): 16
Matriz de features X criada: (2582, 2000)
Vetor de labels Y criado: (2582,)

Dividindo dados (80% treino / 20% teste)...
Tamanho do Treino: 2065
Tamanho do Teste: 517


In [4]:
print("Construindo o modelo CNN-LSTM Híbrido...")

# Usamos +1 no VOCAB_SIZE porque 0 é reservado para o 'padding'
model = Sequential()

# 1. Camada de Embedding
model.add(Embedding(input_dim=VOCAB_SIZE + 1,
                    output_dim=EMBEDDING_DIM,
                    input_length=MAX_SEQ_LENGTH))

# 2. Camada Convolucional (O "Leitor" de Motivos)
# Ele "lê" a sequência e extrai os motivos
model.add(Conv1D(filters=64, kernel_size=10, activation='relu'))
model.add(MaxPooling1D(pool_size=4))
model.add(Dropout(0.4))

# 3. Camada LSTM (O "Leitor" de Ordem)
# O Bidirectional permite que a LSTM leia a sequência de motivos
# da esquerda-para-direita E da direita-para-esquerda
model.add(Bidirectional(LSTM(units=100, return_sequences=False)))
# return_sequences=False significa que ela só retorna o "resumo" final
model.add(Dropout(0.4))

# 4. Camada Densa (Classificador)
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# 5. Camada de Saída
model.add(Dense(1, activation='sigmoid'))

# Compila o modelo
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

Construindo o modelo CNN-LSTM Híbrido...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
print("\nIniciando treinamento da CNN-LSTM...")
print("Usando EarlyStopping: o treino vai parar se a 'val_loss' não melhorar.")

early_stopping = EarlyStopping(monitor='val_loss',
                              patience=3, # Se não melhorar por 3 épocas, pare.
                              restore_best_weights=True)

history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=64,
                    validation_data=(X_test, y_test),
                    callbacks=[early_stopping])

print("Treinamento concluído.")


Iniciando treinamento da CNN-LSTM...
Usando EarlyStopping: o treino vai parar se a 'val_loss' não melhorar.
Epoch 1/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 85ms/step - accuracy: 0.5028 - loss: 0.6956 - val_accuracy: 0.5532 - val_loss: 0.6900
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.4783 - loss: 0.6937 - val_accuracy: 0.5280 - val_loss: 0.6901
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.5269 - loss: 0.6931 - val_accuracy: 0.5261 - val_loss: 0.6895
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 67ms/step - accuracy: 0.5106 - loss: 0.6922 - val_accuracy: 0.5338 - val_loss: 0.6849
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.5353 - loss: 0.6894 - val_accuracy: 0.5010 - val_loss: 0.6944
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.5138 - loss: 0.6933 - val_accuracy: 0.5725 - val_loss: 0.6860
Epoch 7/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.5246 - loss: 0.6914 - val_accuracy: 0.5435 - val_loss: 0.6833
E

In [6]:
print("\nAvaliando modelo final nos dados de teste...")

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"\n--- Relatório de Desempenho (CNN-LSTM FINAL) ---")
print(f"Acurácia Geral: {acc * 100:.2f}%")

print("\nRelatório de Classificação (Precisão, Recall, F1 por Classe):")
print(classification_report(y_test, y_pred))


Avaliando modelo final nos dados de teste...
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step

--- Relatório de Desempenho (CNN-LSTM FINAL) ---
Acurácia Geral: 59.57%

Relatório de Classificação (Precisão, Recall, F1 por Classe):
              precision    recall  f1-score   support

           0       0.61      0.53      0.57       259
           1       0.58      0.66      0.62       258

    accuracy                           0.60       517
   macro avg       0.60      0.60      0.59       517
weighted avg       0.60      0.60      0.59       517

